# 04 — Train and evaluate late-delivery models

Run top to bottom in a **fresh authenticated lakehouse_to_ai AIDP session** on supported Intel/AMD
Spark compute with native Experiments/Models configured. Install wheel
**aidp_lakehouse_to_ai-0.4.0-py3-none-any.whl**, then restart Python. This notebook depends on no
variables or monkeypatches from earlier notebooks. scikit-learn, pandas and the native MLflow integration
must be available. Do not replace AIDP MLflow with a local/custom tracking server.

**User-reported Gold:** reconciliation PASS, identical fingerprints on two executions, zero additional
inserts; 30,600 operations / 296 KPIs / 30,000 training / 600 scoring, including 9,816 late deliveries.
The agent did not execute that AIDP job. **Live AIDP MLflow/training verification remains pending.**

Inputs are immutable-version reads of managed Gold training and scoring tables. Only an aggregate
first acceptance/count is read from the 600 open orders: this notebook never scores them. No layer
is changed. Models/metrics/artifacts are logged to native MLflow; a new selected model version is
registered and reloaded by that exact version. No deployment or agent is created.

See docs/late-delivery-training.md for temporal purge rules, metrics and API prerequisites.

In [ ]:
# One configuration section; non-secret values only. Fixed LF v2 snapshot/reference date.
CONFIG = {
    "source": {
        "catalog": "lakehouse_demo", "namespace": "frgpslcqn7sm",
        "bucket": "CristinaVaras-bucket-AIDP",
        "prefix": "demos/lakehouse-to-ai/source-data/v2/seed-42/reference-2026-10-28",
        "reference_date": "2026-10-28", "seed": 42, "batch_id": "initial",
    },
    "validation_start": "2025-10-01T00:00:00Z",
    "test_start": "2026-04-01T00:00:00Z",  # Also the final training cutoff.
    "seed": 42, "max_training_rows": 50000,
    "experiment_name": "lakehouse_to_ai_late_delivery",
    "registered_model_name": "lakehouse_demo_late_delivery_acceptance",
}
# Re-running configuration revokes earlier success/results.
result = registry = capabilities = data = None

## 1. Runtime/API preflight before fitting

Check package versions, supported pipeline/logging APIs and read-only authenticated tracking/registry
access. Native URI/authentication settings are preserved and never printed. If the runtime does not
support a required API, stop and inspect its native Experiments sample code before proceeding.

In [ ]:
result = registry = capabilities = data = None
try:
    import json
    from aidp_lakehouse import __version__
    from aidp_lakehouse.aidp_bronze import BronzeConfig
    from aidp_lakehouse.gold import GoldConfig
    from aidp_lakehouse.training_contract import TrainingConfig, VERSION, feature_contract
    from aidp_lakehouse.training import temporal_partitions
    from aidp_lakehouse.training_spark import read_training_inputs
    from aidp_lakehouse.training_mlflow import preflight, run_experiment
    if __version__ != VERSION:
        raise RuntimeError(f"Install wheel {VERSION} and restart Python")
    spark.conf.set("spark.sql.session.timeZone", "UTC")
    cfg = TrainingConfig(gold=GoldConfig(BronzeConfig(**CONFIG["source"])),
                         **{k:v for k,v in CONFIG.items() if k != "source"})
    capabilities = preflight(cfg, require_aidp=True)
    print(json.dumps(capabilities, indent=2))
    print(json.dumps(feature_contract(), indent=2))
except Exception as error:
    print(f"TRAINING FAIL [runtime/API preflight]: {error}")
    raise

## 2. Pin Gold inputs and inspect temporal partitions

Tune on acceptance dates before validation_start, using only outcomes strictly known before that
boundary. Validation labels must be known before test_start to support causal selection. These
unavailable boundary labels are explicitly counted as purged, never silently reassigned.
The final model is refitted on pre-test accepted orders with outcomes known strictly before test_start.
Test labels are used only for final retrospective evaluation, never for fitting or selection.
The final cutoff must precede or equal the first open acceptance; otherwise stop.

In [ ]:
result = registry = None
try:
    data, first_open_acceptance, snapshots = read_training_inputs(spark, cfg)
    partitions, partition_report = temporal_partitions(data, cfg, first_open_acceptance)
    print(json.dumps(snapshots, indent=2, default=str))
    preview = {k:dict(v) if isinstance(v,dict) else v for k,v in partition_report.items()}
    preview["test"] = {k:v for k,v in preview["test"].items() if k not in ("positives", "prevalence")}
    preview["test"]["label_statistics"] = "Withheld until final evaluation"
    print(json.dumps(preview, indent=2, default=str))
    del partitions  # Training owns fresh copies; no fitted state comes from this inspection.
except Exception as error:
    print(f"TRAINING FAIL [Gold/temporal inputs]: {error}")
    raise

## 3. Tracked fixed search, validation selection and final test

Compare training-prevalence baseline, two logistic C values and two small forest depths (64 trees).
All fitting, scaling and categorical vocabularies use training only in their respective phase.
Model selection uses validation average precision, then ROC-AUC, then fixed candidate order.
Select threshold by validation F1 over a fixed grid (ties: higher threshold). Freeze both choices.
Refit before test and report the selected model and a same-cutoff prevalence baseline on held-out test.
The owned MLflow run covers fitting through registration and exact-version verification.
No accuracy/precision floor is imposed. Undefined metrics are explicit; zero-division precision/recall use zero.

In [ ]:
result = registry = None
try:
    if capabilities is None:
        raise RuntimeError("Complete native MLflow preflight first")
    result, registry = run_experiment(data, cfg, first_open_acceptance, snapshots, capabilities)
    print("Partitions:", json.dumps(result.report["partitions"], indent=2))
    print("Validation comparison:", json.dumps([
        {"candidate":r["candidate"], "metrics":r["metrics"]}
        for r in result.report["validation_candidates"]], indent=2))
    print("Selected:", result.report["selected_candidate"], "Threshold:", result.report["selected_threshold"])
    print("Held-out test:", json.dumps(result.report["test"], indent=2))
except Exception as error:
    print(f"TRAINING FAIL [fit/evaluate]: {error}")
    raise

## 4. Native MLflow logging, registration and exact-version round trip

Log all candidate validation parameters/metrics, partition sizes/dates/prevalence, final test metrics,
Delta versions, feature/temporal contract, environment and the full selected pipeline (including
feature derivation, preprocessing and decision threshold). Registration creates a new model version;
no existing version/alias is overwritten or deployed. Reload that exact version and compare historical
verification probabilities/decisions. No open order is passed to a model.

In [ ]:
try:
    if result is None or registry is None or not registry["registry_load_verified"]:
        raise RuntimeError("The tracked experiment has not completed registration/version reload")
    print(json.dumps(registry, indent=2))
except Exception as error:
    print(f"TRAINING FAIL [MLflow/registry verification]: {error}")
    raise

## 5. Final result and continuation

In [ ]:
try:
    if result is None or registry is None or not registry["registry_load_verified"]:
        raise RuntimeError("Training and exact registry-version verification are incomplete")
    print("TRAINING PASS: chronological evaluation completed; registered full pipeline reloaded by exact version.")
    print("Use this exact model version and its saved cutoff/threshold in a SEPARATE future scoring notebook.")
    print("Open orders scored in this milestone:", result.report["future_open_orders_scored"])
except Exception as error:
    print(f"TRAINING FAIL [final result]: {error}")
    raise